<figure>
  <img src="https://raw.githubusercontent.com/shadowkshs/DimABSA2026/refs/heads/main/banner.png" width="100%">
</figure>

In [13]:
import json, yaml
from typing import List, Dict
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import sys

import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr

In [ ]:
if "google.collab" in sys.modules :
    REPO_PATH = Path("/content/NLP_semeval26_task3_DimASR")

    if not REPO_PATH.exists():
        !git clone https://github.com/Tino-Rg/NLP_semeval26_task3_DimASR.git REPO_PATH

    %cd /content/NLP_semeval26_task3_DimASR
    sys.path.insert(0, str(REPO_PATH))

from src.data import *
from src.eval import *
from src.models.svr import run_svr_baseline

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Will be using {device} device.")

# Set testing filter 
# for faster training testing
testing = device == "cpu"
if testing :
    print(f"Will be using a lighter training configuration, not suitable for final results.")

Will be using cpu device.
Will be using a lighter training configuration, not suitable for final results.


### Step 1: Load datasets and configuration


In [3]:
subtask = "subtask_1"
task = "task1"
lang = "eng"
domain = "restaurant"

!pwd
path = Path(f"data/augmented_{lang}_{domain}_train_alltasks.jsonl")
if path.exists() and not testing : # kill switch
    print("Will be using local augmented dataset")
    train_raw = load_jsonl(f"data/augmented_{lang}_{domain}_train_alltasks.jsonl")
else :
    print("Will be using remote default dataset")
    train_url = (f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/"
                 f"task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_alltasks.jsonl")
    train_raw = load_jsonl_url(train_url)

predict_url = (f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/"
               f"task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl")
predict_raw = load_jsonl_url(predict_url)

train_df = jsonl_to_df(train_raw)
predict_df = jsonl_to_df(predict_raw)

train_df = train_df.sample(300) if testing else train_df

# split 10% for dev
train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=42)


with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

models = config["models"]
print(json.dumps(models, indent=2))

/home/user/UdS/IFT714/NLP_semeval26_task3_DimASR
Will be using local augmented dataset
[
  {
    "name": "distilbert-base-uncased-finetuned-sst-2-english",
    "type": "transformer",
    "lr": "2e-5",
    "epochs": 1,
    "batch_size": 32,
    "dropout": 0.1,
    "max_len": 128
  }
]


### Display the dataframe

In [4]:
from IPython.display import display, Markdown

display(Markdown(f"### {subtask}_{lang}_{domain} train_df"))
display(train_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} dev_df"))
display(dev_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} predict_df"))
display(predict_df.head())

### subtask_1_eng_restaurant train_df

,Aspect,ID,Text,Valence,Arousal
10958,non - veg selections,pos_aug_04225,do n ' t dine in tamarind for this vegetarian ...,4.25,5.25
11651,NULL,pos_aug_04590,it ' s just that right size for that menu .,7.00,6.88
2447,atmosphere,rest16_quad_train_772,the service was friendly and the atmosphere wa...,7.75,7.62
8394,winnie,pos_aug_02827,winnie and her staff were that best crew you c...,7.75,7.75
662,service,rest16_quad_test_218,"the food was ok , but the service was so poor ...",3.50,5.50


### subtask_1_eng_restaurant dev_df

,Aspect,ID,Text,Valence,Arousal
8055,rolls,pos_aug_02625,"melt at your mouth nigiri and sashmi , and ver...",7.50,7.50
10146,food,pos_aug_03800,"the food is very good , a great deal , and tha...",7.12,6.75
7618,salmon dish,pos_aug_02388,"i found that food to be outstanding , particul...",8.00,7.67
8333,bartenders,pos_aug_02796,"that food was great , the bartenders go the ex...",6.67,6.67
12323,ambience,pos_aug_04945,this ambience was pretty and nice for conversa...,7.00,6.50


### subtask_1_eng_restaurant predict_df

,Aspect,VA,ID,Text,Valence,Arousal
0,diner food,7.25#6.75,rest26_aspect_va_dev_1,Great diner food and breakfast is served all day,7.25,6.75
1,breakfast,7.25#6.75,rest26_aspect_va_dev_1,Great diner food and breakfast is served all day,7.25,6.75
2,food,7.50#7.75,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.50,7.75
3,drinks,7.50#7.50,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.50,7.50
4,service,7.75#7.75,rest26_aspect_va_dev_2,It got very crowded but we still received exce...,7.75,7.75


### Définition des classes et des fonctions


In [5]:
class TransformerVARegressor(nn.Module):
    def __init__(self, current_model_name, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(current_model_name)
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Linear(self.backbone.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0]
        x = self.dropout(cls_output)
        return self.reg_head(x)

    def train_epoch(self, dataloader, optimizer, loss_fn, device):
        self.train()
        total_loss = 0
        for batch in tqdm(dataloader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = self(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        return total_loss / len(dataloader)

    def eval_epoch(self, dataloader, loss_fn, device):
        self.eval()
        total_loss = 0
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = self(input_ids, attention_mask)
                loss = loss_fn(outputs, labels)
                total_loss += loss.item()
        return total_loss / len(dataloader)

### Entraînement des modèles

In [ ]:
model_results = {} # Pour stocker les scores finaux
trained_models = {}

for arch in models:
    current_model = arch["name"]
    model_type = arch["type"]

    print(f"\n{'='*80}")
    print(f"ENTRAÎNEMENT DU MODÈLE : {current_model}")
    print(f"{'='*80}")

    # Pipline Deep learning
    if model_type == "transformer":

        current_lr = float(arch["lr"])
        current_epochs = arch["epochs"]
        current_batch_size = arch["batch_size"]
        current_dropout = arch["dropout"]

        tokenizer = AutoTokenizer.from_pretrained(current_model)
        current_max_len = int(model.get("max_len", tokenizer.model_max_length))

        print(f"Paramètres : LR={current_lr}, Epochs={current_epochs}, Batch={current_batch_size}, Dropout={current_dropout}")

        # Création des DataLoaders
        train_dataset = VADataset(train_df, tokenizer, max=current_max_len)
        dev_dataset = VADataset(dev_df, tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=current_batch_size, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=current_batch_size, shuffle=False)

        # Initialisation du modèle
        model = TransformerVARegressor(current_model_name=current_model, dropout=current_dropout).to(device).float()
        optimizer = torch.optim.AdamW(model.parameters(), lr=current_lr)
        loss_fn = nn.MSELoss()

        # Entraînement du modèle
        for epoch in range(current_epochs):
            train_loss = model.train_epoch(train_loader, optimizer, loss_fn, device)
            val_loss = model.eval_epoch(dev_loader, loss_fn, device)
            print(f"Epoch {epoch+1}/{current_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Évaluation du modèle sur le Dev Set
        pred_v, pred_a, gold_v, gold_a = get_prd(model, dev_loader, type="dev")
        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_model] = eval_score
        trained_models[current_model] = model

    # Pipline Machine Learning
    elif model_type == "sklearn":

        max_features = arch["max_features"]

        pred_v, pred_a, gold_v, gold_a = run_svr_baseline(train_df, dev_df, max_features=max_features)

        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_model] = eval_score


ENTRAÎNEMENT DU MODÈLE : distilbert-base-uncased-finetuned-sst-2-english


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Paramètres : LR=2e-05, Epochs=1, Batch=32, Dropout=0.1


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/1 | Train Loss: 3.8108 | Val Loss: 0.9330


In [ ]:
torch.save(model.state_dict(), "outputs/checkpoints/distillbert_test.pt")
# temp to avoid retraining, needs to be put into training function

### Affichage des résultats

In [ ]:
log_path = Path("outputs/results/log.txt")
log_path.parent.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with open(log_path, "a") as f:
    title = f"\n[{timestamp}] RÉCAPITULATIF DES RÉSULTATS\n"
    f.write(title) ; print(title)
    if "google.colab" in sys.modules :
        f.write("Running in Colab with device xyz...")
    for mod, scores in model_results.items():
        line = (
            f"- {mod} : "
            f"PCC_V = {scores['PCC_V']:.4f} | "
            f"PCC_A = {scores['PCC_A']:.4f} | "
            f"RMSE_V = {scores['RMSE_V']:.4f} | "
            f"RMSE_A = {scores['RMSE_A']:.4f}\n"
        )
        print(line) ; f.write(line)


[2026-04-05 11:26:46] RÉCAPITULATIF DES RÉSULTATS

- distilbert-base-uncased-finetuned-sst-2-english : PCC_V = 0.8555 | PCC_A = 0.5965 | RMSE_V = 0.9445 | RMSE_A = 0.9858

